In [6]:
from dataclasses import dataclass
from typing import Callable, Any, Optional
import torch.nn as nn
import torch
# Attempt relative import first (when running as a package), else fall back to absolute import by adding project root to sys.path
try:
    from ...modules.attention.MQA import MQA, MQAConfig
    from ...modules.normalize.RMSNorm import RMSNorm
except Exception:
    import sys, pathlib
    # try to locate a 'modules' package in the current directory or any parent directory
    cwd = pathlib.Path.cwd()
    modules_root = None
    for p in [cwd] + list(cwd.parents):
        if (p / "modules").is_dir():
            modules_root = p
            break
    if modules_root is None:
        # fallback: add cwd so at least local imports may work
        modules_root = cwd
    modules_root_str = str(modules_root)
    if modules_root_str not in sys.path:
        # put it at front to prefer local package
        sys.path.insert(0, modules_root_str)
    from modules.attention.MQA import MQA, MQAConfig
    from modules.normalize.RMSNorm import RMSNorm

@dataclass
class config:
    vocab :int = 50257
    dim :int = 512
    heads :int = 8
    head_dim :int = 64
    ffn_dim :Optional[int] = 4*dim
    layers :int = 6
    loop :int = 3

class ffn(nn.Module):
    def __init__(self, config:config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.dim, config.ffn_dim),
            nn.GELU(),
            nn.Linear(config.ffn_dim, config.dim)
        )
    def forward(self, x):
        return self.net(x)

class looped_transformer(nn.Module):
    def __init__(self, config:config, MQAconfig:MQAConfig):
        super().__init__()
        # keep reference to config instance
        self.config = config

        self.attn_blocks = nn.ModuleList(
            [nn.Sequential(MQA(config=MQAconfig), RMSNorm(self.config.dim))
             for _ in range(self.config.layers)]
        )
        self.ffn = ffn(self.config)
        # self.ln1 = RMSNorm(self.config.dim)
        self.ln2 = RMSNorm(self.config.dim)

    def forward(self, x):
        for _ in range(self.config.loop):
            for attn_block in self.attn_blocks:
                x = x + attn_block(x)
            x = x + self.ffn(self.ln2(x))
        return x

mqaconfig = MQAConfig(n_embd=512, block_size=1024, n_head=8, kv_head=4, rope=True)
m = looped_transformer(config=config(), MQAconfig=mqaconfig)
